<img src="https://cads.ai/image/CADSLogo.png" style="height: 100px;" align=left>
<div align=right><svg height=100px viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><g clip-path="url(#clip1_4783_495)"><path d="M6.64803 14.103C7.89438 12.8566 8.595 11.1643 8.595 9.40177C8.595 7.63928 7.89378 5.94697 6.64803 4.70058L1.94697 0C0.701225 1.24639 0 2.9387 0 4.70119C0 6.46368 0.701225 8.15599 1.94697 9.40238L6.64742 14.103H6.64803Z" fill="#7B9AB5"></path><path d="M16.4845 14.5379C15.2388 13.2921 13.5459 12.5908 11.7841 12.5908C10.0222 12.5908 8.32936 13.2921 7.08301 14.5379L11.7841 19.239C13.0298 20.4848 14.7227 21.1861 16.4851 21.1861C18.2476 21.1861 19.9398 20.4848 21.1862 19.239L16.4851 14.5379H16.4845Z" fill="#7B9AB5"></path><path d="M1.95832 19.228C3.20468 20.4738 4.89693 21.1751 6.65938 21.1751V14.5269H0.0107422C0.0113472 16.2893 0.711968 17.9817 1.95832 19.228Z" fill="#7B9AB5"></path><path d="M18.2997 7.58717C17.0533 6.34138 15.3611 5.63953 13.598 5.64014C11.8356 5.64014 10.1433 6.34138 8.89697 7.58777L13.598 12.289L18.2997 7.58717Z" fill="#7B9AB5"></path></g></svg></div>

<a box-sizing="border-box" color-scheme=dark font-family="IBM Plex Sans" line-height=30px font-size=72px font-weight=700 align=center>© 2026 — Mehdi Lotfinejad ⨯ Azhar bin Ab Halim @ DATATWEETS</a>

# Trusted Answers: Validate, Repair Once, Critique with a Budget
Day 1 | Notebook 3 of 4

![A conveyor belt carries claim boxes past an inspector with a magnifying glass. A good box gets a tick. A bad box is lifted into a repair bay, repaired once, and returns ticked. A bottom strip shows a draft, critique and revise loop with a stop sign after 2 rounds.](media/d1-03-hero-inspection-belt.svg)

**You can re-run any practice until it works. Nothing here is recorded or marked.**

**Today in one sentence:** You make the helper's answers checkable: a fixed shape, one repair, and a draft that improves within a budget.

## Today's story

The Seri Maju Berhad helpdesk (fictional; Kuala Lumpur, Penang, Johor Bahru) wants claim answers that other programs can read, such as a claims system. A model's free text is not enough. You make the model answer in a fixed shape, **check** it, and **repair** a bad answer once. Then, you improve a draft reply with a **critique loop** that must stop at a budget.

*Amounts are in ringgit (RM) and invented. *Fictional example, not legal or tax advice.*

## What you will be able to do

4. Get a structured answer, validate it with Pydantic, repair it once, and compare it with the source facts.
5. Run a draft, critique and revise loop with a stop budget and read its log.

---

**Prerequisites:** `d1-01-*.ipynb` and `d1-02-*.ipynb` help (done, check, budget; templates).

**Words you do not know?** See the [Glossary](glossary.md).

In [1]:
# Answer switch (change it, then choose Run > Restart Kernel and Run All Cells).
#     False = you type the blanks.
#     True = Run All reads the reference answers.
USE_SOLUTIONS = globals().get("USE_SOLUTIONS", False)

In [2]:
%pip install "langchain-groq>=0.3,<0.4"

In [3]:
# ---- CANONICAL SETUP ----
# ---- HELPER - Run it, then ignore it. ----


## You can re-run any practice until it works. Nothing here is recorded or marked.
# Standard-library helpers for settings files, timing and folder paths.
import json
import os
import sys
import time
from glob import glob
from pathlib import Path
from typing import Literal

# Third-party pieces: HTTP client, Groq chat model, and LangChain base classes for models and embeddings.
import httpx
from dotenv import load_dotenv
from langchain_core.caches import InMemoryCache
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_groq import ChatGroq

# scikit-learn powers the offline TF-IDF embeddings; stop early with an install hint if it is missing.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError as e:
    raise ImportError(
        "scikit-learn is needed for TfidfEmbeddings. In a terminal run: "
        "pip install scikit-learn  (then restart the kernel)."
    ) from e

# Load a default .env file, if there is one, into the environment.
load_dotenv()


SOLUTIONS = {}  # step_id -> (builder function, one-sentence why, code text shown by show_solution)
CHECKPOINTS = {}  # part_id -> function(ns) that rebuilds the variables a late learner needs (canned/mock only)
CODES = {}  # part -> reference code texts, in order

# Look up per-million-token prices for the chosen GROQ_MODEL, used for cost estimates.
# MYR is preferred, then USD, then a built-in default.
with open("_groq_model_per_million_prices.json", mode="rb") as _model_prices_json:
    _MODEL_PRICES = json.load(_model_prices_json).get(
        os.environ.get("GROQ_MODEL", ""), {}
    )
    model_input_price = (_model_input_price := _MODEL_PRICES.get("input", {})).get(
        "MYR", _model_input_price.get("USD", 0.50)
    )
    model_output_price = (_model_output_price := _MODEL_PRICES.get("output", {})).get(
        "MYR", _model_output_price.get("USD", 2.00)
    )


def register(step_id, why, code=""):
    """Decorator used by the notebook: SOLUTIONS['d1-01-p2'] = the reference builder."""

    def deco(fn):
        SOLUTIONS[step_id] = (fn, why, code or getattr(fn, "__doc__", "") or "")
        return fn

    return deco


def checkpoint(part_id):
    """Decorator: register the function that rebuilds this Part's variables (see fast_forward)."""

    def deco(fn):
        CHECKPOINTS[part_id] = fn
        return fn

    return deco


def pick(step_id, attempt=None, is_using_solutions: bool = True):
    """Use your attempt, or the reference answer if USE_SOLUTIONS is True or your attempt is None."""
    if is_using_solutions or attempt is None:
        if step_id not in SOLUTIONS:
            raise KeyError(
                f"No reference answer registered for '{step_id}'. Click the heading 'Part A', "
                "choose Run > Run All Above Selected Cell, then run this cell again."
            )
        return SOLUTIONS[step_id][0]()
    return attempt


def check(step_id, ok, hint=""):
    """Plain self-check: tick or cross plus a hint. Unlimited re-runs. Nothing is stored or counted."""
    print(
        ("\u2714 " + step_id + ": looks right")
        if ok
        else (
            "\u2718 "
            + step_id
            + ": not yet. "
            + (hint or "Re-read the step, change your code, run again.")
        )
    )
    return bool(ok)


def show_solution(step_id):
    """Print the reference code and why it works. Keyboard-friendly: no copying needed, use pick()."""
    if step_id not in SOLUTIONS:
        print("No reference answer registered for", step_id)
        return
    _, why, code = SOLUTIONS[step_id]
    print(code.rstrip() or "(see the reference cell)")
    print("Why:", why)


def fast_forward(part_id):
    """Late or stuck? Rebuild this Part's variables from reference code (canned/mock, no API calls)."""

    if part_id not in CHECKPOINTS:
        print(
            "Not ready yet. Click the heading 'Part A', choose Run > Run All Above Selected Cell, "
            "then run this cell again. Known parts:",
            ", ".join(sorted(CHECKPOINTS)) or "(none registered yet)",
        )
        return

    if globals().get("LLM_MODE") == "groq":
        print("Note: fast-forward uses canned replies, not the live model.")

    CHECKPOINTS[part_id](globals())
    print("\u2714 Caught up to", part_id, "- continue with the next cell.")


# Load the workplace policy texts as system messages so every answer can be grounded in them.
def get_system_corpus(context: Literal["workplace"]) -> list[tuple[str, str]]:
    _corpus_dir: Path
    system_prompt: list[tuple[str, str]] = []

    if context == "workplace":
        _corpus_dir = Path.cwd() / "data" / "workplace*" / "*.txt"

    else:
        raise ValueError(f"Unsupported corpus: '{context}'")

    # Read each policy .txt file under data/workplace*/ as one system message.
    for _corpus_pathname in glob(str(_corpus_dir)):
        with open(_corpus_pathname, "r", encoding="utf-8") as _corpus_file:
            system_prompt.append(("system", _corpus_file.read()))

    # Finish with a rule that makes answers cite their source identifier.
    system_prompt.append(
        (
            "system",
            "Assistant should always cite source identifier to support claim when answering any asks.",
        )
    )

    return system_prompt


# Remember a code snippet under its Part so a late learner can rebuild that Part.
def state(part, code):
    CODES.setdefault(part, []).append(code.strip())


# Register a reference answer: keep its code text and a builder that runs it and returns result.
def ref(part, step, why, code):
    code = code.strip()
    if part:
        CODES.setdefault(part, []).append(code)

    # Run the reference code in a copy of the notebook variables and hand back its result.
    def build():
        ns = dict(globals())
        exec(code, ns)
        return ns["result"]

    register(step, why, code)(build)


# Define how to catch up: re-run the stored code of the listed Parts in order (used by fast_forward).
def rebuild(part, *parts):
    checkpoint(part)(lambda ns: [exec(c, ns) for p in parts for c in CODES[p]])


# Build the shared system messages once so later cells can reuse them.
SYSTEM_PROMPT = get_system_corpus("workplace")


# Guard the Python version: 3.14 or newer is too new for some course packages.
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too new for this course: some packages are not ready. "
        "Create the environment with Python 3.12 (3.11-3.13 work) and pick it as the notebook kernel."
    )

# Also stop on versions older than the supported 3.11.
assert (3, 11) <= sys.version_info[:2] <= (3, 13), (
    f"Use Python 3.11-3.13 (you have {sys.version.split()[0]})"
)
# Key lookup order:
#   1. file named in LM_ENV_FILE
#   2. ~/.langchain-mastery/.env (local, not synced)
#   3. .env next to notebooks.
# Only these four names are read from the file; everything else in it is ignored.

# Names read from the settings file, and folder names that suggest cloud-synced storage.
_ALLOWED = ("GROQ_API_KEY", "GROQ_MODEL", "LLM_MODE", "LM_ENV_FILE")
_SYNCED = ("onedrive", "sharepoint", "dropbox", "google drive", "googledrive", "icloud")


def _tilde(
    p,
):  # show the home folder as ~ so a shared screen does not show your user name
    p, home = str(p), str(Path.home())
    return (
        "~" + p[len(home) :]
        if p.lower().startswith(home.lower())
        else "(outside home) " + Path(p).name
    )


# Warn early if LM_ENV_FILE points to a file that does not exist.
_lm = os.getenv("LM_ENV_FILE", "").strip()
if _lm and not Path(_lm).is_file():
    print(
        "NOTE: LM_ENV_FILE is set but that file was not found, so it is ignored. Check the path."
    )

# Candidate settings files in lookup order; the first one that exists is used.
_env_files = [
    _lm,
    str(Path.home() / ".langchain-mastery" / ".env"),
    str(Path.cwd() / ".env"),
    str(Path.cwd() / "courseware" / ".env"),
]

# Copy only the allowed names from the chosen file into the environment, never overwriting values already set.
_env_file = next((p for p in _env_files if p and Path(p).is_file()), None)
if _env_file:
    try:
        from dotenv import dotenv_values

        for _k, _v in dotenv_values(_env_file).items():
            if (
                _k in _ALLOWED and _v is not None and _k not in os.environ
            ):  # existing variables win
                os.environ[_k] = _v

        # Tell the learner which file was used, with the home folder hidden.
        print(
            "Settings file:",
            _tilde(_env_file),
            "(only",
            ", ".join(_ALLOWED),
            "are read)",
        )

        # Warn if the file sits in a synced folder, where a key could be shared by accident.
        if any(w in _env_file.lower() for w in _SYNCED):
            print(
                "WARNING: your .env is in a synced folder (OneDrive, SharePoint, Dropbox, Google Drive "
                "or iCloud). Move it to ~/.langchain-mastery/.env"
            )

    except ImportError:
        print("python-dotenv not installed: using system environment variables only")
else:
    # With no settings file the notebook still runs in mock mode, so no key is needed.
    print("No .env file found: mock mode, no key needed.")

# Choose mock (offline, canned replies) or groq (live model); mock is the default.
LLM_MODE = os.getenv("LLM_MODE", "mock").strip().lower()
if LLM_MODE not in ("mock", "groq"):
    raise ValueError("LLM_MODE must be 'mock' or 'groq' (check your .env file)")

# Tracing switch: mock mode forces every tracing variable off; groq mode only warns if one is on.
_TRACING = ("LANGCHAIN_TRACING_V2", "LANGCHAIN_TRACING", "LANGSMITH_TRACING")
if LLM_MODE == "mock":
    for _t in _TRACING:
        os.environ[_t] = "false"

    print(
        "Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere)."
    )

elif any(os.getenv(_t, "").strip().lower() == "true" for _t in _TRACING):
    print(
        "WARNING: a LangChain tracing variable is on: live runs may be sent to a tracing service. Set it to false."
    )


class ScriptedToolChat(BaseChatModel):
    """Mock model for tool calling: replays a script. A str is a final answer; a dict
    {'name':..., 'args':{...}} (or a list of them) becomes AIMessage.tool_calls. Canned, not real."""

    # The script to replay and the position of the next step.
    script: list = []
    pos: int = 0

    @property
    def _llm_type(self):
        return "scripted-tool-chat"

    def bind_tools(self, tools, **kwargs):
        return self  # tools are ignored: the script decides

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Take the next scripted step, looping back to the start when the script runs out.
        step = (
            self.script[self.pos % len(self.script)] if self.script else "(mock reply)"
        )
        self.pos += 1

        # A text step becomes a final answer; a dict step becomes tool calls with made-up ids.
        if isinstance(step, str):
            msg = AIMessage(content=step)

        else:
            calls = step if isinstance(step, list) else [step]
            msg = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": c["name"],
                        "args": c.get("args", {}),
                        "id": f"mock_{self.pos}_{i}",
                        "type": "tool_call",
                    }
                    for i, c in enumerate(calls)
                ],
            )

        return ChatResult(generations=[ChatGeneration(message=msg)])


def get_llm(responses=None, tool_calls=None):
    """Return a chat model. Groq in 'groq' mode; offline canned mock otherwise."""

    if LLM_MODE == "groq":
        # Live mode needs both a model name and a key; otherwise explain what is missing.
        model = os.getenv("GROQ_MODEL", "").strip()

        if not model or not os.getenv("GROQ_API_KEY", "").strip():
            raise RuntimeError(
                "Groq mode needs GROQ_API_KEY and GROQ_MODEL in .env "
                "(suggested model: openai/gpt-oss-20b; confirm in the Groq console), "
                "or set LLM_MODE=mock."
            )

        # Return the Groq model; temperature 0 and a cache keep replies repeatable.
        print("LIVE model:", model)
        return ChatGroq(
            cache=InMemoryCache(),
            model=model,
            temperature=0,
            http_client=httpx.Client(verify=False),
        )

    # Mock mode: an offline fake model, with scripted tool calls if given, otherwise canned text replies.
    print("MOCK: canned replies")
    if tool_calls:
        return ScriptedToolChat(script=list(tool_calls))

    return FakeListChatModel(responses=list(responses or ["(mock reply)"]))


class TraceLog:
    """Tiny local trace: one dict per step. This class sends nothing anywhere."""

    RATE_PER_1K_TOKENS = 0.0005  # PLACEHOLDER rate in RM per 1,000 tokens (not a real price); use your plan's

    def __init__(self):
        self.rows = []

    def add(self, step, tool="-", tokens_in=0, tokens_out=0, seconds=0.0):
        # Estimate cost from the total tokens at the placeholder rate (not a real price).
        cost = (tokens_in + tokens_out) / 1000 * self.RATE_PER_1K_TOKENS

        # Store one row per step so the trace can be shown as a table.
        self.rows.append(
            {
                "step": step,
                "tool": tool,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "seconds": round(seconds, 3),
                "est_cost_myr": round(cost, 6),
            }
        )

    def timed(self, step, fn, *args, tool="-", **kw):
        # Run fn, time it, and log a step; token counts are rough guesses (about 4 characters per token).
        t0 = time.perf_counter()
        out = fn(*args, **kw)

        self.add(
            step,
            tool,
            len(str(args)) // 4,
            len(str(out)) // 4,
            time.perf_counter() - t0,
        )

        return out

    # Sum tokens, seconds and estimated cost over all recorded steps.
    def total(self):
        return {
            k: round(sum(r[k] for r in self.rows), 6)
            for k in ("tokens_in", "tokens_out", "seconds", "est_cost_myr")
        }


class TfidfEmbeddings(Embeddings):
    """Offline embeddings: scikit-learn TF-IDF fitted on your corpus, L2-normalised."""

    def __init__(self, corpus):
        # Fit the vocabulary on the corpus once so documents and questions share one vector space.
        self.vec = TfidfVectorizer(norm="l2", lowercase=True).fit(list(corpus))

    def embed_documents(self, texts):
        return self.vec.transform(list(texts)).toarray().tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


# ---- END HELPER ----
# ---- END CANONICAL SETUP ----

No .env file found: mock mode, no key needed.
Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere).


## Three rules

1. **Keep your key private.** Never paste it into a notebook, chat or email.
2. **Do not paste personal or company data.** Live mode sends text to a third party (Groq). Use only this course's fictional data.
3. **AI output needs a human check.** A neat answer can still be wrong.

Mock mode (the default) needs no key.

**Practices:** a blank starter looks like `my_attempt = None`. **If you leave it as `None`, the notebook quietly runs the reference answer instead, so Run All works even when you wrote nothing.**

To learn, write your own and set `my_attempt` to it: then yours runs. `check(...)` prints ✔ or ✘ and a hint. `show_solution(step_id)` prints the reference code.

* `USE_SOLUTIONS`: `False` = you type the blanks,
* `USE_SOLUTIONS`: `True` = Run All reads the reference answers.

Reading first, then trying your own, teaches more.

In [4]:
# Tools for this notebook: Pydantic checks the reply shape; prompts, parser and Literal build the rest.
import json
from typing import Literal

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field

# Confirm the imports ran and show which model mode (mock or live) is active.
print("LLM_MODE =", LLM_MODE, "| imports ready")

LLM_MODE = mock | imports ready


In [5]:
# Register how to rebuild each part's variables, so a late learner can catch up with fast_forward.
rebuild("d1-03-A", "d1-03-A")
rebuild("d1-03-B", "d1-03-A", "d1-03-B")

In [6]:
# Store the claims source of truth as code text so a checkpoint can rebuild it later.
state(
    "d1-03-A",
    "CLAIMS = [dict(id=i, item=a, amount=m, site=s, action=x) for i, a, m, s, x in [\n"
    "    ('C1', 'taxi to a client site', 35.0, 'Kuala Lumpur', 'approve'),\n"
    "    ('C2', 'laptop repair', 1400.0, 'Penang', 'ask_first'),\n"
    "    ('C3', 'customer dinner', 3800.0, 'Johor Bahru', 'ask_first'),\n"
    "    ('C4', 'alcohol at a team event', 210.0, 'Kuala Lumpur', 'reject')]]\n"
    "TEXTS = [c['item'] + ', RM ' + format(c['amount'], ',.0f') + ', ' + c['site']\n"
    "         for c in CLAIMS]",
)

# Store the reference parser for the Step 2 practice (cut out the JSON, then validate).
ref(
    "d1-03-A",
    "d1-03-parse",
    "Cut out the JSON, load it, then validate it.",
    "def parse_note(text):\n"
    "    data = json.loads(text[text.find('{'):text.rfind('}') + 1])\n"
    "    return ClaimNote.model_validate(data)\n"
    "result = parse_note",
)

# Store the schema and the helpers that give a good reply or the error text.
state(
    "d1-03-A",
    "class ClaimNote(BaseModel):\n"
    "    item: str\n"
    "    amount_myr: float = Field(ge=0, le=100000)\n"
    "    site: Literal['Kuala Lumpur', 'Penang', 'Johor Bahru']\n"
    "    action: Literal['approve', 'ask_first', 'reject']\n"
    "def good(c, **over):\n"
    "    keys = dict(item='item', amount_myr='amount', site='site', action='action')\n"
    "    return json.dumps({**{k: c[v] for k, v in keys.items()}, **over})\n"
    "def safe(fn, *args):\n"
    "    try:\n"
    "        return fn(*args)\n"
    "    except ValueError as err:\n"
    "        return str(err)[:200]",
)

# Store the ask prompt and the repair prompt used by the generate and fix chains.
state(
    "d1-03-A",
    "gen_prompt = ChatPromptTemplate.from_messages([\n"
    "    ('system', 'Reply with one JSON object. Keys: item, amount_myr, site, action.'),\n"
    "    ('human', 'Claim text: {text}')])\n"
    "fix_prompt = ChatPromptTemplate.from_messages([\n"
    "    ('system', 'Fix the JSON so the check accepts it. Reply with JSON only.'),\n"
    "    ('human', 'Bad reply: {bad} Check error: {error}')])",
)

# Store the reference check that compares a valid note with the source claim.
ref(
    "d1-03-B",
    "d1-03-matches",
    "Valid is not right: compare with the source.",
    "def matches_source(note, claim):\n"
    "    return (note.amount_myr == claim['amount'] and note.site == claim['site']\n"
    "            and note.action == claim['action'])\n"
    "result = matches_source",
)

# Store the reference repair function: ask once, repair once, then give up.
ref(
    "d1-03-A",
    "d1-03-repair",
    "One repair call at most, then give up.",
    "def get_note(gen, fix, text):\n"
    "    raw = gen.invoke({'text': text})\n"
    "    note = safe(parse_note, raw)\n"
    "    if isinstance(note, ClaimNote):\n"
    "        return note, 'ok'\n"
    "    fixed = safe(parse_note, fix.invoke({'bad': raw, 'error': note}))\n"
    "    if isinstance(fixed, ClaimNote):\n"
    "        return fixed, 'repaired'\n"
    "    return None, 'gave_up'\n"
    "result = get_note",
)

# Store the facts and the code check that defines a good reply.
state(
    "d1-03-B",
    "FACTS = ('Claim: laptop repair, RM 1,400, Penang. Policy SM-POL-03: over RM 500 '\n"
    "         'needs department head approval.')\n"
    "def check_reply(text):\n"
    "    need = [('RM 1,400', 'the amount'), ('department head', 'the approver')]\n"
    "    problems = ['too long'] if len(text.split()) > 30 else []\n"
    "    return problems + ['missing ' + w for k, w in need if k.lower() not in text.lower()]",
)

# Store the canned draft and revisions that stand in for a model.
state(
    "d1-03-B",
    "D0 = ('Thank you for asking about the laptop repair in Penang. We looked at the '\n"
    "      'rules and there are several steps you must follow before any money can be '\n"
    "      'paid to you.')\n"
    "R1 = 'Your RM 1,400 laptop repair claim needs one more approval before payment.'\n"
    "R2 = 'Your RM 1,400 repair claim needs department head approval, then Finance pays.'",
)

# Store the builder of fresh canned chains (draft, critique, revise).
state(
    "d1-03-B",
    "def make_chains():\n"
    "    def link(t, r):\n"
    "        llm = get_llm(responses=r)\n"
    "        return PromptTemplate.from_template(t) | llm | StrOutputParser()\n"
    "    return (link('Write a short reply from these facts: {facts}', [D0]),\n"
    "            link('Draft: {draft} Problems: {problems}. One short fix instruction.',\n"
    "                 ['Shorten it and add the amount.', 'Name who must approve it.']),\n"
    "            link('Draft: {draft} Fix: {note}. Rewrite in at most 30 words.', [R1, R2]))",
)

# Store the reference helper that runs the loop with a chosen budget.
ref(
    "d1-03-B",
    "d1-03-budget",
    "Fresh chains each run; the budget caps the loop.",
    "def run_with_budget(n):\n"
    "    return critique_loop(*make_chains(), max_revisions=n)\n"
    "result = run_with_budget",
)

# Store the reference critique loop: stop on done or budget, else critique and revise.
ref(
    "d1-03-B",
    "d1-03-loop",
    "Stop on done or budget, else critique + revise.",
    "def critique_loop(draft_c, critique_c, revise_c, max_revisions=3):\n"
    "    log, draft = [], draft_c.invoke({'facts': FACTS})\n"
    "    for rev in range(max_revisions + 1):\n"
    "        problems = check_reply(draft)\n"
    "        log.append(dict(check=rev + 1, problems=problems))\n"
    "        if not problems:\n"
    "            return draft, log, 'done'\n"
    "        if rev == max_revisions:\n"
    "            return draft, log, 'budget'\n"
    "        args = {'draft': draft, 'problems': '; '.join(problems)}\n"
    "        draft = revise_c.invoke({'draft': draft, 'note': critique_c.invoke(args)})\n"
    "result = critique_loop",
)

## Validate, then repair once

### Idea: Structured Output, the Inspection Belt (●)

- A program cannot rely on free text. **Structured output** means the answer has fixed fields, such as `item`, `amount_myr`, `site` and `action`.
- A **schema** is the written description of that shape. **Pydantic** is the Python third-party library we use to write it and to **validate** (check) a reply against it.
- **Validate, then repair once.** Ask. Check the reply. If it is bad, ask again **once**, giving the model its own bad reply and the error. If it is still bad, stop and say so: a human looks. The belt in the picture does exactly this.
- **Valid is not the same as right.** A reply can have the right shape and a wrong amount. So we also compare with the source facts.

> ~**MOCK warning.**
> All model replies in this notebook are canned, and some are bad on purpose, so you can see the repair path.
> A live model will not behave like this script.

**Step 1 - the facts and the prompts (●)**

Run this cell as it is. `CLAIMS` is the source of truth for our checks. `TEXTS` are what staff typed. The two prompts are one to ask and one to repair.

In [7]:
# The source of truth for every later check: four fictional claims with the correct values.
CLAIMS = [
    dict(id=i, item=a, amount=m, site=s, action=x)
    for i, a, m, s, x in [
        ("C1", "taxi to a client site", 35.0, "Kuala Lumpur", "approve"),
        ("C2", "laptop repair", 1400.0, "Penang", "ask_first"),
        ("C3", "customer dinner", 3800.0, "Johor Bahru", "ask_first"),
        ("C4", "alcohol at a team event", 210.0, "Kuala Lumpur", "reject"),
    ]
]

TEXTS = [
    c["item"] + ", RM " + format(c["amount"], ",.0f") + ", " + c["site"] for c in CLAIMS
]  # CLAIMS: the source of truth for our checks (fictional)

# The ask prompt: it tells the model which JSON keys to return.
gen_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Reply with one JSON object. Keys: item, amount_myr, site, action."),
        ("human", "Claim text: {text}"),
    ]
)

# The repair prompt: it gives the model its own bad reply and the check error.
fix_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Fix the JSON so the check accepts it. Reply with JSON only."),
        ("human", "Bad reply: {bad} Check error: {error}"),
    ]
)

# Show the claim texts staff would type.
print(TEXTS)

['taxi to a client site, RM 35, Kuala Lumpur', 'laptop repair, RM 1,400, Penang', 'customer dinner, RM 3,800, Johor Bahru', 'alcohol at a team event, RM 210, Kuala Lumpur']


In [8]:
class ClaimNote(BaseModel):  # the schema: the shape every reply must have
    item: str
    amount_myr: float = Field(ge=0, le=100000)
    site: Literal["Kuala Lumpur", "Penang", "Johor Bahru"]
    action: Literal["approve", "ask_first", "reject"]


def good(c, **over):  # a correct JSON reply built from a claim row
    keys = dict(item="item", amount_myr="amount", site="site", action="action")

    return json.dumps({**{k: c[v] for k, v in keys.items()}, **over})


def safe(fn, *args):  # run fn; give back its result, or the error text if it raised
    try:
        return fn(*args)

    except ValueError as err:
        return str(err)[:200]


# Try a bad note (negative amount) to see the validator reject it with a readable error.
bad = dict(item="x", amount_myr=-5, site="Penang", action="approve")
print(safe(ClaimNote.model_validate, bad))

1 validation error for ClaimNote
amount_myr
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-5, input_type=int]
    For further information visit https://errors.pyda


In [9]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-groq>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langgraph>=0.6,<0.7" "pydantic>=2,<3" "scikit-learn>=1.4" "python-dotenv"

In [10]:
import sys
import langchain
import langchain_core
import langchain_groq

print("Python:", sys.version)
print("LangChain:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Groq:", getattr(langchain_groq, "__version__", "unknown"))
print("LangChain file:", langchain.__file__)
print("Has verbose:", hasattr(langchain, "verbose"))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
LangChain: 0.3.30
LangChain Core: 0.3.86
LangChain Groq: 0.3.8
LangChain file: /usr/local/lib/python3.13/dist-packages/langchain/__init__.py
Has verbose: True


In [11]:
print("DATA:", "DATA" in globals())
print("get_llm:", "get_llm" in globals())
print("pick:", "pick" in globals())
print("check:", "check" in globals())
print("USE_SOLUTIONS:", "USE_SOLUTIONS" in globals())

DATA: False
get_llm: True
pick: True
check: True
USE_SOLUTIONS: True


**Step 2 - parse a reply (●)**

Models often add words around the JSON. Cut out the text from the first `{` to the last `}`, load it, and validate it. Bad input must raise an error, not go unnoticed.

In [12]:
# Practice: cut the JSON out of a chatty reply, load it, then validate it.
def my_parse(text):
    start, end = text.find("{"), text.rfind("}")

    data = None  # TODO: json.loads of text[start:end + 1]

    return ClaimNote.model_validate(data)


my_attempt = None  # when ready, write: my_attempt = my_parse

# Use your parser, or the reference one if you left the blank empty.
parse_note = pick("d1-03-parse", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Test with a chatty reply, text with no JSON and an empty object; bad input must give an error.
reply = "Sure! " + good(CLAIMS[0])

ok = isinstance(safe(parse_note, reply), ClaimNote)
ok = ok and isinstance(safe(parse_note, "no json here"), str)
ok = ok and isinstance(safe(parse_note, "{}"), str)
print(safe(parse_note, "no json here"))

# Self-check: good input parses and bad input is rejected.
check("d1-03-parse", ok, "Parse from the first { to the last }; bad text must raise.")

Expecting value: line 1 column 1 (char 0)
✔ d1-03-parse: looks right


True

**Step 3 - repair once (●)**

`gen` and `fix` are *chains*: `a | b | c` means feed the output of `a` into `b`, then into `c`. You will meet it again in the stretch.

Complete the function: ask once, and if the reply is bad, ask the fix chain **once**. The status is `ok`, `repaired` or `gave_up`.

In [13]:
# Practice: ask once, and if the reply is bad, repair it once, then give up.
def my_get_note(gen, fix, text):
    # First try: ask the generate chain for a note.
    raw = gen.invoke({"text": text})

    # A valid first reply is returned as it is.
    note = safe(parse_note, raw)
    if isinstance(note, ClaimNote):
        return note, "ok"

    # TODO: ask the fix chain ONCE: fix.invoke({'bad': raw, 'error': note}); parse it

    # Still bad after the one repair: stop and leave it for a human.
    return None, "gave_up"


my_attempt = None  # when ready, write: my_attempt = my_get_note

# Use your function, or the reference one if you left the blank empty.
get_note = pick("d1-03-repair", my_attempt, is_using_solutions=USE_SOLUTIONS)
print("get_note is ready:", callable(get_note))

get_note is ready: True


### Check yourself

1. Why validate a reply that looks like JSON? <details><summary>Answer:</br></br></summary>It may have a wrong value, a missing field or extra words. The check catches that before a program uses it.</br></br></details>
2. Why repair only once? <details><summary>Answer:</br></br></summary>Repeated repairs can run on and cost money. One try, then stop and let a human look. That is the stop budget again.</br></br></details>


To continue you need `CLAIMS`, `ClaimNote`, `parse_note` and `get_note`.

In [14]:
# Part B needs the Part A names; list any that are missing.
need = ["CLAIMS", "ClaimNote", "parse_note", "get_note", "safe"]
missing = [n for n in need if n not in globals()]

print("All set for Part B." if not missing else "Missing: " + ", ".join(missing))

# Catch up from the reference code if Part A was skipped.
if missing and "d1-03-A" not in CODES:
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d1-03-A")  # rebuilds the names above from reference code

# Confirm that the parser accepts a correct reply.
ok = isinstance(safe(parse_note, good(CLAIMS[1])), ClaimNote)
print("parse_note works:", ok)

All set for Part B.
parse_note works: True


## Right, not just valid; then the critique loop

### Warm-up from Part A: run four claims, and valid is not the same as right (●)

**Practice - valid is not the same as right (●)**

1. First write `matches_source`.
2. Then the next two cells run four claims through canned replies and show a table.

Look for the claim that is *valid* but does *not match* the source.

In [15]:
# Practice: valid is not right, so compare the note with the source claim.
def my_matches(note, claim):
    return None  # TODO: True when amount_myr, site and action equal the claim's values


my_attempt = None  # when ready, write: my_attempt = my_matches

# Use your check, or the reference one if you left the blank empty.
matches_source = pick("d1-03-matches", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Build one correct note and one that is valid but has the wrong amount.
note = ClaimNote.model_validate_json(good(CLAIMS[0]))
wrong = ClaimNote.model_validate_json(good(CLAIMS[0], amount_myr=36))

# Self-check: the correct note matches the source and the wrong one does not.
check(
    "d1-03-matches",
    matches_source(note, CLAIMS[0]) and not matches_source(wrong, CLAIMS[0]),
    "Compare amount_myr, site and action with the source claim.",
)

✔ d1-03-matches: looks right


True

In [16]:
# Canned model replies for four claims: good, invalid, valid but wrong, and no JSON.
GEN_MOCK = [
    "Sure! " + good(CLAIMS[0]),
    good(CLAIMS[1], action="maybe"),
    good(CLAIMS[2], amount_myr=380.0),
    "Not sure about this one.",
]

FIX_MOCK = [good(CLAIMS[1]), "I still cannot tell."]  # canned: 2nd repair is bad

# Build the ask and repair chains on canned models so the demo repeats without an API key.
gen = gen_prompt | get_llm(responses=GEN_MOCK) | StrOutputParser()
fix = fix_prompt | get_llm(responses=FIX_MOCK) | StrOutputParser()

# Run every claim through ask and repair, then compare each note with its source.
RESULTS = [(c, *get_note(gen, fix, t)) for c, t in zip(CLAIMS, TEXTS)]
print("claim  status    against the source")

# Show each status and whether the answer matches the source.
for c, note, status in RESULTS:
    same = note is not None and matches_source(note, c)

    word = "no answer" if note is None else "matches" if same else "does not match"
    print(c["id"], "   ", f"{status:<9}", word)

MOCK: canned replies
MOCK: canned replies
claim  status    against the source
C1     ok        matches
C2     repaired  matches
C3     ok        does not match
C4     gave_up   no answer


In [17]:
# Turn the statuses into three rates: valid first time, valid after one repair, and right.
st = [s for _, _, s in RESULTS]

first = st.count("ok") / len(st)

after = (st.count("ok") + st.count("repaired")) / len(st)

right = sum(n is not None and matches_source(n, c) for c, n, _ in RESULTS) / len(st)

msg = f"valid first time {first:.0%} | valid after one repair {after:.0%}"
print(msg, f"| right {right:.0%}")

# Self-check: the statuses match the expected outcome.
if LLM_MODE == "mock":
    check(
        "d1-03-repair",
        st == ["ok", "repaired", "ok", "gave_up"],
        "Repair only once; a second bad reply means gave_up.",
    )

else:
    check("d1-03-repair", set(st) <= {"ok", "repaired", "gave_up"}, "Use 3 statuses.")

valid first time 50% | valid after one repair 75% | right 50%
✔ d1-03-repair: looks right


### Idea: Draft, Critique, Revise, and Stop (●)

A **critique loop** is a finishable loop from d1-01 with a model in it. The model writes a **draft**. A plain **check** (code, not opinion) lists the problems. If there are problems, the model **critiques** the draft (one short fix instruction) and then **revises** it. Then we check again.

The **revision budget** is the maximum number of revisions. When it is used up, the loop stops, even if the draft is still not good.

Our task: write a reply to a staff member about a RM 1,400 laptop repair claim. **Done means:** at most 30 words, it says the amount, and it names who must approve.

**Step 1 - the facts, the check and the canned chains (●)**

Run both cells as they are. Read `check_reply`: it is the definition of done as code.

In mock mode the first draft is too long and has no amount, the first revision still lacks the approver, and the second revision is good.

In [18]:
# The facts the reply must be built from.
FACTS = (
    "Claim: laptop repair, RM 1,400, Penang. Policy SM-POL-03: over RM 500 "
    "needs department head approval."
)


def check_reply(text):  # the check: a list of problems; an empty list means done
    problems = []

    if len(text.split()) > 30:
        problems.append("too long")

    if "RM 1,400" not in text:
        problems.append("missing the amount")

    if "department head" not in text.lower():
        problems.append("missing the approver")

    return problems


# Try the check on a bad reply to see the problems it lists.
print(check_reply("Please wait."))

['missing the amount', 'missing the approver']


In [19]:
# Canned draft and revisions: the draft is too long with no amount; the last revision is good.
D0 = (
    "Thank you for asking about the laptop repair in Penang. We looked at the "
    "rules and there are several steps you must follow before any money can be "
    "paid to you."
)
R1 = "Your RM 1,400 laptop repair claim needs one more approval before payment."
R2 = "Your RM 1,400 repair claim needs department head approval, then Finance pays."


def make_chains():  # fresh canned models each time (canned replies replay in order)

    def link(t, r):
        llm = get_llm(responses=r)
        return PromptTemplate.from_template(t) | llm | StrOutputParser()

    return (
        link("Write a short reply from these facts: {facts}", [D0]),
        link(
            "Draft: {draft} Problems: {problems}. One short fix instruction.",
            ["Shorten it and add the amount.", "Name who must approve it."],
        ),
        link("Draft: {draft} Fix: {note}. Rewrite in at most 30 words.", [R1, R2]),
    )


# Show the problems found in the draft and in each revision.
print([check_reply(x) for x in (D0, R1, R2)])

[['too long', 'missing the amount', 'missing the approver'], ['missing the approver'], []]


**Step 2 - the loop (●)**

Fill the three TODOs: stop when there are no problems; stop when the budget is used up; otherwise critique and revise.

Read the log it prints.

In [20]:
# Practice: build the critique loop: check, then critique and revise until done or out of budget.
def my_loop(draft_c, critique_c, revise_c, max_revisions=3):
    log, draft = [], draft_c.invoke({"facts": FACTS})

    # Check the draft once per round, with one extra check after the last revision.
    for rev in range(max_revisions + 1):
        problems = check_reply(draft)

        log.append(dict(check=rev + 1, problems=problems))

        # TODO 1: no problems -> return draft, log, 'done'
        # TODO 2: rev == max_revisions -> return draft, log, 'budget'
        # TODO 3: ask critique_c, then revise_c, and keep the new draft

    return draft, log, "budget"


my_attempt = None  # when ready, write: my_attempt = my_loop

# Use your loop, or the reference one if you left the blanks empty.
critique_loop = pick("d1-03-loop", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Run the loop with fresh canned chains and a budget of 3.
final, log, status = critique_loop(*make_chains(), max_revisions=3)
print("status:", status, "| final:", final)

# Read the log: which problems each check found.
for row in log:
    print("check", row["check"], "|", "; ".join(row["problems"]) or "no problems")

MOCK: canned replies
MOCK: canned replies
MOCK: canned replies
status: done | final: Your RM 1,400 repair claim needs department head approval, then Finance pays.
check 1 | too long; missing the amount; missing the approver
check 2 | missing the approver
check 3 | no problems


**Practice - three budgets (●)**

Write a small helper that runs the loop with a given budget and fresh chains. Then the next cell tries budgets 0, 1 and 3. The loop may log at most *budget + 1* checks.

In [21]:
# Practice: a helper that runs the loop with fresh chains and a given budget.
def my_run(n):
    return None  # TODO: run critique_loop with fresh make_chains() and max_revisions=n


my_attempt = None  # when ready, write: my_attempt = my_run

# Use your helper, or the reference one if you left the blank empty.
run_with_budget = pick("d1-03-budget", my_attempt, is_using_solutions=USE_SOLUTIONS)

EXPECT = {0: ("budget", 1), 1: ("budget", 2), 3: ("done", 3)}  # canned outcomes
ok = True

# Try budgets 0, 1 and 3 and compare the result with what is expected.
for n in (0, 1, 3):
    out = run_with_budget(n)

    final, log, status = out if out else ("", [], "no result")
    print("budget", n, "->", status, "| checks logged:", len(log))

    ok = ok and len(log) <= n + 1 and (status == "done") == (check_reply(final) == [])

    if LLM_MODE == "mock":
        ok = ok and (status, len(log)) == EXPECT[n]

# Self-check: the log never exceeds budget + 1 checks.
check("d1-03-budget", ok, "At most budget + 1 checks; done only if the check is clean.")

MOCK: canned replies
MOCK: canned replies
MOCK: canned replies
budget 0 -> budget | checks logged: 1
MOCK: canned replies
MOCK: canned replies
MOCK: canned replies
budget 1 -> budget | checks logged: 2
MOCK: canned replies
MOCK: canned replies
MOCK: canned replies
budget 3 -> done | checks logged: 3
✔ d1-03-budget: looks right


True

### Check yourself

1. Claim C3 was valid. Was it right? <details><summary>Answer:</br></br></summary>No. The amount was RM 380 instead of RM 3,800. Valid means the right shape; right means it matches the source.</br></br></details>
2. What should happen to a `gave_up` claim? <details><summary>Answer:</br></br></summary>A person handles it. The helper must not guess.</br></br></details>
3. What stops the loop? <details><summary>Answer:</br></br></summary>Either the check finds no problems (done), or the revision budget is used up.</br></br></details>
4. With budget 3, at which check number did the problems list first become empty? <details><summary>Answer:</br></br></summary>Check 3 (draft, then revision 1, then revision 2).</br></br></details>
5. Why is the check code and not the model's own opinion? <details><summary>Answer:</br></br></summary>Code gives the same answer every time and cannot be talked into approving a bad draft.</br></br></details>
6. Budget 1 ended as `budget`. What should happen next? <details><summary>Answer:</br></br></summary>A person reads the last draft. The loop must not pretend it succeeded.</br></br></details>

You should have `matches_source`, `check_reply`, `make_chains`, `critique_loop` and `run_with_budget`.

In [22]:
# Confirm that every helper from this notebook exists.
need = [
    "matches_source",
    "check_reply",
    "make_chains",
    "critique_loop",
    "run_with_budget",
]
missing = [n for n in need if n not in globals()]

print("d1-03 complete." if not missing else "Missing: " + ", ".join(missing))

# Catch up from the reference code if something is missing.
if missing and not all(p in CODES for p in ("d1-03-A", "d1-03-B")):
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d1-03-B")  # rebuilds Part A and Part B names from reference code

# Final run with budget 3 to see the loop finish.
final, log, status = run_with_budget(3) or ("", [], "no result")
print("budget 3 ->", status, "after", len(log), "checks")

d1-03 complete.
MOCK: canned replies
MOCK: canned replies
MOCK: canned replies
budget 3 -> done after 3 checks


## Recap and next step

- **Validate, then repair once.** Parse, validate, repair once, and give up out loud. Valid is not the same as right: compare with the source.
- **A critique loop is a finishable loop.** Check with code, revise, and stop at the budget. Read the log.
- All model replies here were canned. A live model needs the same checks, and a human still reads what matters.
- Next: **[d1-04-meaning-map-and-fifa-brief.ipynb](d1-04-meaning-map-and-fifa-brief.ipynb)** (find the right policy by meaning, then build the checked brief).

### A schema variant and a sequential chain (◆)

The next cell does both. First it adds a field to the schema with inheritance and shows the validator rejecting a bad value. Then it builds a **sequential chain**: the output of the first model call becomes the input of the second.

Read the printed result.

In [23]:
class UrgentNote(ClaimNote):  # same fields plus one more
    urgency: Literal["low", "high"]


# Show the validator rejecting a bad value in the extended schema.
base = json.loads(good(CLAIMS[0]))
print(safe(UrgentNote.model_validate, {**base, "urgency": "soon"}))

# Two prompts: the first pulls out key points, the second rewrites them as one sentence.
outline = PromptTemplate.from_template("List 2 key points from: {facts}")
short = PromptTemplate.from_template("Rewrite as one sentence: {points}")

# Reuse one canned model for both steps.
llm = get_llm(responses=["1. RM 1,400 repair. 2. Needs approval.", R2])

# Sequential chain: the output of the first call becomes the input of the second.
seq = {"points": outline | llm | StrOutputParser()} | short | llm | StrOutputParser()

# Run the chain and read the one-sentence result.
print(seq.invoke({"facts": FACTS}))

1 validation error for UrgentNote
urgency
  Input should be 'low' or 'high' [type=literal_error, input_value='soon', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/
MOCK: canned replies
Your RM 1,400 repair claim needs department head approval, then Finance pays.


### Notes for later (■)

- **Extra fields.** Add `claim_id` and `receipt: bool` to `ClaimNote`.

  Which canned replies now break validation, and what do the error messages say?

- **Parser variants.** LangChain can also ask a model for structured output directly (`with_structured_output`) or parse with `PydanticOutputParser`. Both still need the same checks. Read about them after class.
- **Honesty labels.** All replies in this notebook are canned. The claim policy figures are fictional (`data/workplace_corpus/sm-pol-03-expense-claims.txt`). Not legal or tax advice.